# Qt–Ft Agglomeration Simulation — Run Notebook

Coarse-grained ReaDDy2 Brownian-dynamics simulation of Qt encapsulin and ferritin (Ft) agglomeration.

**This notebook only runs simulations.** All parameters live in the single **Configuration** cell; the single **Run** cell then executes any combination of:

- `RUN_MODE = "single"` (one trajectory) or `"ensemble"` (multi-replica)
- `ENABLE_DEAGG = False` (plain agglomeration) or `True` (agglomeration ↔ deagglomeration cycling)

Plotting and analysis live in the separate `Plot_Ensemble_Results_*.ipynb` notebooks.

## 1. Imports and Setup

In [3]:
import os
import sys
import readdy

# qtft package. Plotting lives in the Plot_Ensemble_Results_* notebooks, so it is
# deliberately not imported here; `analysis` is only used for the optional summary/XYZ export.
import qtft as sim
import qtft.analysis as analysis
from qtft import EnsembleSimulation

print(f"ReaDDy: {readdy.__version__}")
print(f"Python: {sys.version}")

ReaDDy: 2.0.13-5
Python: 3.10.19 | packaged by conda-forge | (main, Oct 22 2025, 22:29:10) [GCC 14.3.0]


## 2. Module Reload Helper

In [4]:
import importlib
import qtft, qtft.config, qtft.system, qtft.engine, qtft.analysis, qtft.ensemble

# Reload submodules in dependency order (restart the kernel for deep changes).
for _m in (qtft.config, qtft.system, qtft.engine, qtft.analysis, qtft.ensemble, qtft):
    importlib.reload(_m)

sim = qtft
analysis = qtft.analysis
from qtft import EnsembleSimulation
print("Modules reloaded")

Modules reloaded


## 3. Configuration — all parameters

In [ ]:
# ============================================================
# SIMULATION CONFIGURATION  —  all parameters live here
# ============================================================
# Edit parameters, run this cell, then run the "Run" cell below.
#
# >>> SOFT / large-timestep preset <<<  (see README section 12)
# Configured for a soft potential ("soft" or "weak") so a large integration timestep is
# numerically stable. This is a HEAVILY COARSE-GRAINED regime — keep in mind:
#   * Reachable time = N_STEPS x TIMESTEP;  config.print_summary() below prints the total.
#   * Diffusion is lowered so the per-step displacement sqrt(2*D*dt) stays << the
#     particle / binding-radius scales; "reachable time" is MODEL time under that
#     slow-particle assumption, not literal wall-clock or real-particle time.
#   * k_bond and the soft/weak force constants must stay soft enough for the chosen dt
#     (k * D must stay small). Calibrate with:  python scripts/calibrate_timestep.py
#   * kon is capped by p = 1 - exp(-kon*dt): a rate that saturates can't be resolved at large dt.
#   * equilibration_potential must be "soft"/"weak" at large dt (WCA/LJ would blow up).
# TIP: pilot with a small N_STEPS first (confirm stability + that clusters nucleate).

# ----- What to run -----
RUN_MODE     = "single"    # "single" (one trajectory) or "ensemble" (multi-replica)
ENABLE_DEAGG = False       # False = plain agglomeration; True = agglomeration <-> deagglomeration cycling

# ----- Potential -----
# Pick ONE production potential here. Only the matching parameter block in the config below
# is registered; the others (lj epsilons / soft.k_* / weak.k_*,depth_*) are ignored.
POTENTIAL = "weak"    # "soft" (harmonic repulsion) | "weak" (soft attractive) | "LJ" | "WCA"

# ----- Time & steps -----
TIMESTEP = 1e3        # ns   integration timestep (1e3 ns = 1 µs)
N_STEPS  = 500000     # steps for a plain run (ignored when ENABLE_DEAGG). Total time = N_STEPS x TIMESTEP.

# ----- Deagglomeration / cycling steps (used when ENABLE_DEAGG) -----
KOFF            = 1e-7        # per-edge bond-breaking rate (1/ns); tuned so p_break stays stochastic
AGG_STEPS       = 150_000     # steps of each agglomeration phase
DEAGG_STEPS     = 150_000     # steps of each deagglomeration phase
N_CYCLES        = 1           # number of agg->deagg cycles (total phases = 2 * N_CYCLES)
AGG_POTENTIAL   = "soft"      # potential during agglomeration   ("soft" / "weak" / "LJ" / "WCA")
DEAGG_POTENTIAL = "soft"      # potential during deagglomeration

# ----- Output locations & toggles -----
SINGLE_RUN_ROOT = "Simulation_Files_Single_Runs"  # parent folder for single runs
ENSEMBLE_ROOT   = "Simulation_Files_Ensembles"    # parent folder for ensembles
SAVE_CONFIG     = True    # write <name>_config.json next to the trajectory (ensembles always save it)
EXPORT_XYZ      = True    # also export an OVITO-friendly .xyz after the run

# ----- Ensemble settings (used when RUN_MODE == "ensemble") -----
N_REPLICAS          = 10
PARALLEL            = True
N_WORKERS           = 10
EQUILIBRATION_STEPS = 1000    # reaction-free relaxation before production (runs under equilibration_potential)

# ============================================================
# Build the configuration
# ============================================================
config = sim.SimulationConfig(
    # ----- Potential selector (set POTENTIAL at the top) -----
    potential_type=POTENTIAL,    # picks which block below is registered; the others are ignored

    # ----- Particle properties -----
    qt=sim.ParticleConfig(
        name="Qt",
        radius=25.0,             # nm (encapsulin)
        diffusion=2e-4,          # nm^2/ns
        cluster_diffusion=2e-5,  # nm^2/ns (when bound in a cluster)
    ),
    ft=sim.ParticleConfig(
        name="Ft",
        radius=7.0,              # nm (ferritin)
        diffusion=5e-4,          # nm^2/ns
        cluster_diffusion=5e-5,  # nm^2/ns (when bound in a cluster)
    ),

    # ----- Topology / binding -----
    topology=sim.TopologyConfig(
        name="QtFt_Cluster",
        binding_radius=32.0,     # nm (~ r_Qt + r_Ft + buffer)
        kon=1e-8,                # microscopic binding rate (1/ns)
        k_bond=1.0,              # kJ/(mol*nm^2) bond stiffness (keep soft for large dt)
        ft_monovalent=False,     # True -> Ft caps at 1 bond (single-Qt-star clusters); adds _FtMono tag
        allow_loops=False,       # True -> intra-cluster crosslinks/loops (networked, not trees); needs ft_monovalent=False; adds _loops tag
    ),

    # ----- Parameters for potential_type="LJ"/"WCA" (epsilons; ignored otherwise) -----
    lj=sim.LennardJonesConfig(
        epsilon_QtQt=1.5,
        epsilon_FtFt=1.5,
        epsilon_QtFt=3.0,
    ),

    # ----- Parameters for potential_type="soft" (per-pair harmonic repulsion; ignored otherwise) -----
    # k in kJ/(mol*nm^2); 0 disables a pair. Overlap scale ~ sqrt(2*kB*T/k): stiffen the small Ft
    # (k_FtFt / k_QtFt) to reduce overlap. Cluster/mixed pairs cascade from these three unless set.
    soft=sim.SoftPotentialConfig(
        k_QtQt=0.1,   # Qt-Qt
        k_FtFt=0.1,   # Ft-Ft  (raise to stiffen small-particle repulsion / reduce overlap)
        k_QtFt=0.1,   # Qt-Ft  (raise to stiffen)
    ),

    # ----- Parameters for potential_type="weak" (piecewise-harmonic; ignored otherwise) -----
    # Soft ATTRACTIVE alternative to LJ: a well of depth `depth` with its minimum at contact
    # (r_i+r_j), returning to 0 at cutoff = cutoff_factor * contact. Per-pair k (branch stiffness)
    # + depth (attraction); both cascade like soft/LJ; k=0 disables a pair.
    weak=sim.WeakInteractionConfig(
        k_QtQt=0.5,      k_FtFt=3.0,      k_QtFt=2.0,       # kJ/(mol*nm^2) branch stiffness
        depth_QtQt=0.25,  depth_FtFt=0.1,  depth_QtFt=8.0,   # kJ/mol attraction (cross stronger)
        cutoff_factor=1.1,                                  # cutoff = 2.0 * contact
    ),

    # ----- Equilibration (reactions always off) -----
    equilibration_potential="soft",   # must be "soft"/"weak" at large dt (WCA/LJ would blow up)

    # ----- Simulation box -----
    box_size=(500, 500, 500),    # nm
    periodic_boundary=True,
    temperature=300.0,           # K

    # ----- Integration & recording -----
    timestep=TIMESTEP,           # ns  (set in "Time & steps" above)
    n_steps=N_STEPS,             # steps (set above; ignored when ENABLE_DEAGG)
    record_stride=100,                  # save trajectory every N steps
    observable_stride=100,              # record observables every N steps
    particles_observable_stride=None,   # redundant with the trajectory: structural analysis reads positions
                                        # from it (record_stride). Set an int only to speed up per-frame
                                        # structural analysis, at the cost of storing positions twice.

    # ----- Particle counts -----
    n_qt=150,
    n_ft=300,

    # ----- Execution -----
    kernel="CPU",
    n_threads=4,
    rng_seed=22,

    # output_file is auto-generated from the parameters (see qtft.format_param_string).
)

# ----- Apply deagglomeration cycling (centralizes the phase knobs set above) -----
if ENABLE_DEAGG:
    config.topology.koff = KOFF
    config.phases = sim.make_agg_deagg_phases(
        agg_steps=AGG_STEPS,
        deagg_steps=DEAGG_STEPS,
        n_cycles=N_CYCLES,
        agg_potential=AGG_POTENTIAL,
        deagg_potential=DEAGG_POTENTIAL,
    )

config.print_summary()

## 4. Run

In [ ]:
# ============================================================
# RUN  —  dispatches on RUN_MODE and ENABLE_DEAGG (set in the config cell)
# ============================================================

# Auto-named basename from the parameters (idempotent: safe to re-run this cell).
base_name = sim.format_param_string(config) + ".h5"

if RUN_MODE == "single":
    # Collect all output for this run in its own subfolder under SINGLE_RUN_ROOT.
    RUN_DIR = os.path.join(SINGLE_RUN_ROOT, base_name[:-3])
    config.output_file = os.path.join(RUN_DIR, base_name)

    result = sim.run_one(config, equilibration_steps=EQUILIBRATION_STEPS)

    if config.phases:
        # Phased run: result is a list of per-phase dicts; the stitched whole-cycle
        # trajectory path is under result[0]["combined"].
        combined = result[0].get("combined")
        export_src = combined
        print(f"Phased run complete ({len(result)} phases). Combined trajectory: {combined}")
    else:
        # Plain run: result is a readdy.Trajectory.
        export_src = config.output_file
        analysis.print_analysis_summary(config.output_file, config)

    # Overlap check: minimum center-to-center distance per species pair (final frame).
    if export_src and os.path.exists(export_src):
        analysis.print_min_distance_table(export_src, config)

    if SAVE_CONFIG:
        cfg_path = config.output_file[:-3] + "_config.json"
        config.save_json(cfg_path)
        print(f"Saved config: {cfg_path}")

    if EXPORT_XYZ and export_src and os.path.exists(export_src):
        xyz_path = export_src.replace(".h5", ".xyz")
        analysis.convert_h5_to_xyz(export_src, xyz_path, config, overwrite=True)
        print(f"Exported XYZ: {xyz_path}")

elif RUN_MODE == "ensemble":
    ensemble = EnsembleSimulation(
        base_config=config,
        n_replicas=N_REPLICAS,
        base_dir=ENSEMBLE_ROOT,
    )
    print(f"Seeds: {ensemble.seeds}")
    ensemble.run_local(
        parallel=PARALLEL,
        n_workers=N_WORKERS,
        overwrite=True,
        equilibration_steps=EQUILIBRATION_STEPS,
    )
    ensemble.print_summary()

    # Overlap check for replica_000: minimum center-to-center distance per species pair.
    rep0 = ensemble.replica_configs[0]
    rep0_src = (os.path.join(rep0.phase_base_dir, "trajectory_combined.h5")
                if config.phases else rep0.output_file)
    if os.path.exists(rep0_src):
        print("Overlap (replica_000):")
        analysis.print_min_distance_table(rep0_src, rep0)

    if EXPORT_XYZ:
        rep_cfg = ensemble.replica_configs[0]
        if config.phases:
            src = os.path.join(rep_cfg.phase_base_dir, "trajectory_combined.h5")
        else:
            src = rep_cfg.output_file
        if os.path.exists(src):
            xyz_path = src.replace(".h5", ".xyz")
            analysis.convert_h5_to_xyz(src, xyz_path, rep_cfg, overwrite=True)
            print(f"Exported XYZ (replica_000): {xyz_path}")

else:
    raise ValueError(f"RUN_MODE must be 'single' or 'ensemble', got {RUN_MODE!r}")

## 5. Cluster (SLURM) execution — optional

For HPC runs, generate SLURM job-array scripts instead of running locally. This builds an `EnsembleSimulation` from the **Configuration** cell above (set the params there first), then writes `submit_ensemble.slurm` and `submit_analysis.slurm` into the ensemble directory. Nothing runs locally — submit them with `sbatch`.

In [ ]:
# ============================================================
# OPTIONAL: generate SLURM scripts for cluster execution
# ============================================================
# Builds the ensemble from the config above; does not run anything locally.
ensemble = EnsembleSimulation(
    base_config=config,
    n_replicas=N_REPLICAS,
    base_dir=ENSEMBLE_ROOT,
)

# ----- SLURM script for running replica simulations -----
ensemble.generate_slurm_scripts(
    # --- SLURM job settings ---
    partition="cm4_tiny",        # (required, str) SLURM partition name
    cluster="cm4",               # (optional, str) SLURM cluster name, None to omit
    qos="cm4_tiny",              # (optional, str) Quality of service, None to omit
    time="08:00:00",             # (optional, str) Wall time limit per replica (HH:MM:SS)
    cpus_per_task=12,            # (optional, int) CPUs per replica
    memory="32G",                # (optional, str) Memory per replica

    # --- Conda environment ---
    conda_base="<YOUR_CONDA_PATH>",  # (required, str) Full path to conda installation, e.g. "/home/user/miniconda3"
    conda_env="readdy",              # (optional, str) Name of conda environment with ReaDDy

    # --- Paths ---
    scripts_dir="~/Readdy_Simulations",  # (optional, str) Directory where Python scripts are located

    # --- Email notifications ---
    mail_user=None,              # (optional, str) Email for notifications, e.g. "user@example.com"
    mail_type="ALL",             # (optional, str) When to send emails: NONE, BEGIN, END, FAIL, ALL
)

# ----- SLURM script for post-simulation analysis -----
ensemble.generate_analysis_slurm_script(
    # --- SLURM job settings ---
    partition="cm4_tiny",        # (required, str) SLURM partition name
    cluster="cm4",               # (optional, str) SLURM cluster name, None to omit
    qos="cm4_tiny",              # (optional, str) Quality of service, None to omit
    time="04:00:00",             # (optional, str) Wall time limit (HH:MM:SS)
    cpus_per_task=4,             # (optional, int) CPUs for parallel analysis
    memory="32G",                # (optional, str) Memory allocation

    # --- Conda environment ---
    conda_base="<YOUR_CONDA_PATH>",  # (required, str) Full path to conda installation, e.g. "/home/user/miniconda3"
    conda_env="readdy",              # (optional, str) Name of conda environment with ReaDDy

    # --- Paths and analysis settings ---
    scripts_dir="~/Readdy_Simulations",  # (optional, str) Directory where Python scripts are located
    stride=10,                           # (optional, int) Analyze every Nth frame for structural analysis

    # --- Email notifications ---
    mail_user=None,              # (optional, str) Email for notifications, e.g. "user@example.com"
    mail_type="ALL",             # (optional, str) When to send emails: NONE, BEGIN, END, FAIL, ALL
)